In [1]:
# Tabellen fuer die TVD-Analyse bei p > 2 erstellen
import pandas as pd
from pathlib import Path

from efficient_probit_regression.total_variation_distance import total_variation_distance

dt = pd.read_pickle("Plots/probability_distributions_p_greater_2.pkl")

rows = []

for _, row in dt.sort_values(["dataset", "p"]).iterrows():
    lev = row["prob_lp"]
    lw = row["prob_lewis_weight"]
    rnd = row["prob_random_evaluations_probabilities"]
    scaled_l2lp = row["prob_l2_lp_leverage_score_scaled"]

    rows.append({
        "dataset": row["dataset"],
        "p": float(row["p"]),
        "lp - lewis": round(total_variation_distance(lev, lw), 3),
        "lp - scaled_l2lp": round(total_variation_distance(lev, scaled_l2lp), 3),
        "lp - random": round(total_variation_distance(lev, rnd), 3),
        "scaled_l2lp - lewis": round(total_variation_distance(scaled_l2lp, lw), 3),
        "lewis - random": round(total_variation_distance(lw, rnd), 3),
        "scaled_l2lp - random": round(total_variation_distance(scaled_l2lp, rnd), 3),
    })

dt = pd.DataFrame(rows).sort_values(["p", "dataset"]).reset_index(drop=True)

out_dir = Path("Tabellen")
out_dir.mkdir(exist_ok=True, parents=True)
out_csv = "tvd_all_methods_p_groesser_2.csv"
dt.to_csv(out_dir / out_csv, index=False, float_format="%.10f")

print("Saved:", out_csv)
print("TVD Analyse fuer p > 2:")
for p_value, df_p in dt.groupby("p", sort=False):
    print(f"\n=== p = {p_value} ===")
    print(df_p.reset_index(drop=True).to_string())


Saved: tvd_all_methods_p_groesser_2.csv
TVD Analyse fuer p > 2:

=== p = 2.3 ===
     dataset    p  lp - lewis  lp - scaled_l2lp  lp - random  scaled_l2lp - lewis  lewis - random  scaled_l2lp - random
0  Covertype  2.3       0.546             0.545        0.701                0.042           0.424                 0.413
1  Example2D  2.3       0.137             0.138        0.259                0.031           0.209                 0.214
2       Iris  2.3       0.229             0.226        0.228                0.028           0.177                 0.171
3     KDDCup  2.3       0.426             0.431        0.479                0.032           0.360                 0.362
4    Webspam  2.3       0.669             0.670        0.790                0.046           0.396                 0.397

=== p = 2.5 ===
     dataset    p  lp - lewis  lp - scaled_l2lp  lp - random  scaled_l2lp - lewis  lewis - random  scaled_l2lp - random
0  Covertype  2.5       0.533             0.530        0.723  

In [2]:
# top_k Ergebnisse fuer p > 2
import pandas as pd

print("Ergebnisse von top_k fuer p > 2:")
top_k = pd.read_csv("Plots/BA_Comparison_p_greater_2/TopK_thresholds_table_p_greater_2.csv")
for dataset_name, p_value in top_k.sort_values(["dataset", "p"]).groupby("dataset", sort=False):
    print(f"\n=== Datensatz: {dataset_name} ===")
    for p_val, dataset_new_name in p_value.sort_values(["p", "dataset"]).groupby("p", sort=False):
        print(f"\n====== p = {p_val} ======")
        print(dataset_new_name.reset_index(drop=True).to_string())


Ergebnisse von top_k fuer p > 2:

=== Datensatz: Covertype ===

====== p = 2.3 ======
     dataset    p              method     k50     k90     k95
0  Covertype  2.3  lp Leverage Scores    1238  161498  263137
1  Covertype  2.3       Lewis Weights   48806  374167  456442
2  Covertype  2.3  Random Evaluations  164772  462571  515887
3  Covertype  2.3    d^(p/2-1)L2 + Lp   53421  384722  464399

====== p = 2.5 ======
     dataset    p              method     k50     k90     k95
0  Covertype  2.5  lp Leverage Scores    2493  159598  267826
1  Covertype  2.5       Lewis Weights   46811  368185  451732
2  Covertype  2.5  Random Evaluations  154358  455392  511316
3  Covertype  2.5    d^(p/2-1)L2 + Lp   53448  384736  464428

====== p = 3.0 ======
     dataset    p              method     k50     k90     k95
0  Covertype  3.0  lp Leverage Scores    1321   69182  118805
1  Covertype  3.0       Lewis Weights   34777  347471  436185
2  Covertype  3.0  Random Evaluations  129721  436029  498718


In [3]:
# Bestimmung von c = max(np.maximum(lewis_prob / scaled_l2_lp_prob, scaled_l2_lp_prob / lewis_prob))
import pandas as pd

bestimmung_c = pd.read_csv("Tabellen/Bestimmung_c_symmetrisch_p_groesser_2.csv")
print("Bestimmung von c = max(np.maximum(lewis_prob / scaled_l2_lp_prob, scaled_l2_lp_prob / lewis_prob)):")

for dataset_name, p_value in bestimmung_c.sort_values(["dataset", "p"]).groupby("dataset", sort=False):
    print(f"\n=== Datensatz: {dataset_name} ===")
    print(p_value.reset_index(drop=True).to_string())


Bestimmung von c = max(np.maximum(lewis_prob / scaled_l2_lp_prob, scaled_l2_lp_prob / lewis_prob)):

=== Datensatz: Covertype ===
     dataset    p   d  scale_factor            c
0  Covertype  2.3  55        1.8241      62.6160
1  Covertype  2.5  55        2.7233     578.3221
2  Covertype  3.0  55        7.4162    2055.2367
3  Covertype  3.5  55       20.1963    2096.9814
4  Covertype  4.0  55       55.0000  104818.6909

=== Datensatz: Example2D ===
     dataset    p  d  scale_factor       c
0  Example2D  2.3  3        1.1791  1.1586
1  Example2D  2.5  3        1.3161  1.2763
2  Example2D  3.0  3        1.7321  1.5952
3  Example2D  3.5  3        2.2795  1.9874
4  Example2D  4.0  3        3.0000  2.4646

=== Datensatz: Iris ===
  dataset    p  d  scale_factor       c
0    Iris  2.3  5        1.2731  1.2070
1    Iris  2.5  5        1.4953  1.3726
2    Iris  3.0  5        2.2361  1.8947
3    Iris  3.5  5        3.3437  2.6233
4    Iris  4.0  5        5.0000  3.6358

=== Datensatz: KDDCup 

In [4]:
# Bestimmung von c = max(lewis_prob / scaled_l2_lp_prob)
import pandas as pd

bestimmung_c = pd.read_csv("Tabellen/Bestimmung_c_max_p_groesser_2.csv")
print("Bestimmung von c = max(lewis_prob / scaled_l2_lp_prob):")

for dataset_name, p_value in bestimmung_c.sort_values(["dataset", "p"]).groupby("dataset", sort=False):
    print(f"\n=== Datensatz: {dataset_name} ===")
    print(p_value.reset_index(drop=True).to_string())


Bestimmung von c = max(lewis_prob / scaled_l2_lp_prob):

=== Datensatz: Covertype ===
     dataset    p   d  scale_factor          c
0  Covertype  2.3  55        1.8241   131.1668
1  Covertype  2.5  55        2.7233    71.8375
2  Covertype  3.0  55        7.4162   591.3019
3  Covertype  3.5  55       20.1963   600.1313
4  Covertype  4.0  55       55.0000  2573.1901

=== Datensatz: Example2D ===
     dataset    p  d  scale_factor       c
0  Example2D  2.3  3        1.1791  1.1664
1  Example2D  2.5  3        1.3161  1.2882
2  Example2D  3.0  3        1.7321  1.5868
3  Example2D  3.5  3        2.2795  1.9119
4  Example2D  4.0  3        3.0000  2.2588

=== Datensatz: Iris ===
  dataset    p  d  scale_factor       c
0    Iris  2.3  5        1.2731  1.1264
1    Iris  2.5  5        1.4953  1.2071
2    Iris  3.0  5        2.2361  1.4267
3    Iris  3.5  5        3.3437  1.6562
4    Iris  4.0  5        5.0000  1.8937

=== Datensatz: KDDCup ===
  dataset    p   d  scale_factor        c
0  KDDCup 

In [5]:
# Vergleich von 0.5 * min(d^(p/2-1) * l_2, l_p) <= d^(p/2-1) * w_i <= 2 * max(d^(p/2-1) * l_2, l_p)
import pandas as pd

vergleich_scaled_l2lp = pd.read_csv("Tabellen/Vergleich_minmax_l2_lp_vs_lewis_p_groesser_2.csv")
print("Vergleich von 0.5 * min(d^(p/2-1) * l_2, l_p) <= d^(p/2-1) * w_i <= 2 * max(d^(p/2-1) * l_2, l_p):")

for p_value, dataset_name in vergleich_scaled_l2lp.sort_values(["p", "dataset"]).groupby("p", sort=False):
    print(f"\n=== p = {p_value} ===")
    print(dataset_name.reset_index(drop=True).to_string())


Vergleich von 0.5 * min(d^(p/2-1) * l_2, l_p) <= d^(p/2-1) * w_i <= 2 * max(d^(p/2-1) * l_2, l_p):

=== p = 2.3 ===
     dataset    p    d  scale_factor  true_scores  false_scores  true_scores_percent  true_prob  false_prob  true_prob_percent
0  Covertype  2.3   55        1.8241       580989            23                100.0     580988          24              100.0
1  Example2D  2.3    3        1.1791          175             0                100.0        175           0              100.0
2       Iris  2.3    5        1.2731          150             0                100.0        150           0              100.0
3     KDDCup  2.3   34        1.6972       494021             0                100.0     494015           6              100.0
4    Webspam  2.3  129        2.0729       350000             0                100.0     349994           6              100.0

=== p = 2.5 ===
     dataset    p    d  scale_factor  true_scores  false_scores  true_scores_percent  true_prob  false_pr

In [6]:
# Bestimme die maximale Abweichung von d^(p/2-1) * lewis zu max(d^(p/2-1) * l_2, l_p)
import pandas as pd

vergleich_c_lewis_scaled_l2lp = pd.read_csv("Tabellen/Maximale_Abweichung_lewis_max_l2_lp_p_groesser_2.csv")
print("Bestimme die maximale Abweichung von d^(p/2-1) * lewis zu max(d^(p/2-1) * l_2, l_p):")

for p_value, dataset_name in vergleich_c_lewis_scaled_l2lp.sort_values(["p", "dataset"]).groupby("p", sort=False):
    print(f"\n=== p = {p_value} ===")
    print(dataset_name.reset_index(drop=True).to_string())


Bestimme die maximale Abweichung von d^(p/2-1) * lewis zu max(d^(p/2-1) * l_2, l_p):

=== p = 2.3 ===
     dataset    p    d  scale_factor  max_scores_ratio  max_prob_ratio
0  Covertype  2.3   55        1.8241           80.2697         79.2809
1  Example2D  2.3    3        1.1791            1.1639          1.1639
2       Iris  2.3    5        1.2731            1.1220          1.1021
3     KDDCup  2.3   34        1.6972            1.4420          1.4168
4    Webspam  2.3  129        2.0729            1.6863          1.6863

=== p = 2.5 ===
     dataset    p    d  scale_factor  max_scores_ratio  max_prob_ratio
0  Covertype  2.5   55        2.7233          118.5985        115.3715
1  Example2D  2.5    3        1.3161            1.2792          1.1964
2       Iris  2.5    5        1.4953            1.2063          1.1774
3     KDDCup  2.5   34        2.4147            1.8025          1.8025
4    Webspam  2.5  129        3.3701            2.3475          2.3475

=== p = 3.0 ===
     dataset

In [7]:
# Unnormierte Score-Massen fuer p > 2
import pandas as pd

score_massen = pd.read_csv("Tabellen/Score_Massen_p_groesser_2.csv")
print("Unnormierte Score-Massen fuer p > 2:")

for dataset_name, p_value in score_massen.sort_values(["dataset", "p"]).groupby("dataset", sort=False):
    print(f"\n=== Datensatz: {dataset_name} ===")
    print(p_value.reset_index(drop=True).to_string())


Unnormierte Score-Massen fuer p > 2:

=== Datensatz: Covertype ===
     dataset    p   d  scale_factor     sum_lewis  sum_scaled_lewis  sum_scaled_l2lp  reference_d  reference_d_p_half
0  Covertype  2.3  55        1.8241  5.570150e+01      1.016064e+02         100.3976         55.0            100.3268
1  Covertype  2.5  55        2.7233  5.577570e+01      1.518923e+02         149.8370         55.0            149.7798
2  Covertype  3.0  55        7.4162  5.937400e+01      4.403290e+02         407.8955         55.0            407.8909
3  Covertype  3.5  55       20.1963  5.974840e+01      1.206697e+03        1110.8299         55.0           1110.7970
4  Covertype  4.0  55       55.0000  3.499796e+06      1.924888e+08        3025.0027         55.0           3025.0000

=== Datensatz: Example2D ===
     dataset    p  d  scale_factor  sum_lewis  sum_scaled_lewis  sum_scaled_l2lp  reference_d  reference_d_p_half
0  Example2D  2.3  3        1.1791     3.0000            3.5374           3.5547 